Noise Robustness Evaluation for LS-EMVAE on Task-1, Task-2, Task-3


In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as nnf
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, matthews_corrcoef

def set_seed(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
TASK_CONFIGS = {
    "Task-1": {
        "lead_file_paths": {
            "LEAD_I": "../../Data_processing/data_aspire_PAP/LEAD_I.pt",
            "LEAD_II": "../../Data_processing/data_aspire_PAP/LEAD_II.pt",
            "LEAD_III": "../../Data_processing/data_aspire_PAP/LEAD_III.pt",
            "LEAD_aVR": "../../Data_processing/data_aspire_PAP/LEAD_aVR.pt",
            "LEAD_aVL": "../../Data_processing/data_aspire_PAP/LEAD_aVL.pt",
            "LEAD_aVF": "../../Data_processing/data_aspire_PAP/LEAD_aVF.pt",
            "LEAD_V1": "../../Data_processing/data_aspire_PAP/LEAD_V1.pt",
            "LEAD_V2": "../../Data_processing/data_aspire_PAP/LEAD_V2.pt",
            "LEAD_V3": "../../Data_processing/data_aspire_PAP/LEAD_V3.pt",
            "LEAD_V4": "../../Data_processing/data_aspire_PAP/LEAD_V4.pt",
            "LEAD_V5": "../../Data_processing/data_aspire_PAP/LEAD_V5.pt",
            "LEAD_V6": "../../Data_processing/data_aspire_PAP/LEAD_V6.pt"
        },
        "labels_file_path": "../../Data_processing/data_aspire_PAP/labels.pt"
    },
    "Task-2": {
        "lead_file_paths": {
            "LEAD_I": "../../Data_processing/data_aspire_PAWP/LEAD_I.pt",
            "LEAD_II": "../../Data_processing/data_aspire_PAWP/LEAD_II.pt",
            "LEAD_III": "../../Data_processing/data_aspire_PAWP/LEAD_III.pt",
            "LEAD_aVR": "../../Data_processing/data_aspire_PAWP/LEAD_aVR.pt",
            "LEAD_aVL": "../../Data_processing/data_aspire_PAWP/LEAD_aVL.pt",
            "LEAD_aVF": "../../Data_processing/data_aspire_PAWP/LEAD_aVF.pt",
            "LEAD_V1": "../../Data_processing/data_aspire_PAWP/LEAD_V1.pt",
            "LEAD_V2": "../../Data_processing/data_aspire_PAWP/LEAD_V2.pt",
            "LEAD_V3": "../../Data_processing/data_aspire_PAWP/LEAD_V3.pt",
            "LEAD_V4": "../../Data_processing/data_aspire_PAWP/LEAD_V4.pt",
            "LEAD_V5": "../../Data_processing/data_aspire_PAWP/LEAD_V5.pt",
            "LEAD_V6": "../../Data_processing/data_aspire_PAWP/LEAD_V6.pt"
        },
        "labels_file_path": "../../Data_processing/data_aspire_PAWP/labels.pt"
    },
    "Task-3": {
        "lead_file_paths": {
            "LEAD_I": "D:/ukbiobank/ECG_PAWP_UKB_Final/LEAD_I.pt",
            "LEAD_II": "D:/ukbiobank/ECG_PAWP_UKB_Final/LEAD_II.pt",
            "LEAD_III": "D:/ukbiobank/ECG_PAWP_UKB_Final/LEAD_III.pt",
            "LEAD_aVR": "D:/ukbiobank/ECG_PAWP_UKB_Final/LEAD_aVR.pt",
            "LEAD_aVL": "D:/ukbiobank/ECG_PAWP_UKB_Final/LEAD_aVL.pt",
            "LEAD_aVF": "D:/ukbiobank/ECG_PAWP_UKB_Final/LEAD_aVF.pt",
            "LEAD_V1": "D:/ukbiobank/ECG_PAWP_UKB_Final/LEAD_V1.pt",
            "LEAD_V2": "D:/ukbiobank/ECG_PAWP_UKB_Final/LEAD_V2.pt",
            "LEAD_V3": "D:/ukbiobank/ECG_PAWP_UKB_Final/LEAD_V3.pt",
            "LEAD_V4": "D:/ukbiobank/ECG_PAWP_UKB_Final/LEAD_V4.pt",
            "LEAD_V5": "D:/ukbiobank/ECG_PAWP_UKB_Final/LEAD_V5.pt",
            "LEAD_V6": "D:/ukbiobank/ECG_PAWP_UKB_Final/LEAD_V6.pt"
        },
        "labels_file_path": "D:/ukbiobank/ECG_PAWP_UKB_Final/labels.pt"
    }
}

NOISE_LEVELS = {
    "Mild": {"SNR": 20.0, "ABW": 0.05, "APL": 0.02},
    "Medium": {"SNR": 10.0, "ABW": 0.15, "APL": 0.05},
    "Severe": {"SNR": 5.0, "ABW": 0.30, "APL": 0.10}
}

F_BW = 0.33
F_PL = 50.0
FS = 500.0
PRETRAINED_MODEL_PATH = "pretrain/LS_EMVAE_with_reg_12_lead.pth"
LEADS_6 = ["LEAD_I", "LEAD_II", "LEAD_III", "LEAD_aVR", "LEAD_aVF", "LEAD_aVL"]


In [ ]:
def prior_expert(size, use_cuda=False):
    mu = torch.zeros(size)
    logvar = torch.zeros(size)
    if use_cuda:
        mu, logvar = mu.cuda(), logvar.cuda()
    return mu, logvar

class ProductOfExperts(nn.Module):
    def forward(self, mus, logvars, eps=1e-8):
        var = torch.exp(logvars) + eps
        T = 1. / var
        mu_poe = torch.sum(mus * T, dim=0) / torch.sum(T, dim=0)
        var_poe = 1. / torch.sum(T, dim=0)
        logvar_poe = torch.log(var_poe + eps)
        return mu_poe, logvar_poe

def mixture_of_experts(mus, logvars, weights=None):
    num_experts = mus.shape[0]
    if weights is None:
        weights = torch.ones(num_experts, device=mus.device) / num_experts
    weights = weights / torch.sum(weights)
    weights = weights.view(num_experts, 1, 1)
    combined_mu = torch.sum(weights * mus, dim=0)
    combined_logvar = torch.sum(weights * logvars, dim=0)
    return combined_mu, combined_logvar

class ECGLeadEncoder(nn.Module):
    def __init__(self, input_dim=5000, latent_dim=256):
        super(ECGLeadEncoder, self).__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1)
        self.flatten = nn.Flatten()
        conv_output_dim = 5000 // (2 ** 3)
        self.fc_mu = nn.Linear(64 * conv_output_dim, latent_dim)
        self.fc_logvar = nn.Linear(64 * conv_output_dim, latent_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.flatten(x)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

class SharedDecoder(nn.Module):
    def __init__(self, latent_dim=256, output_dim=5000):
        super(SharedDecoder, self).__init__()
        self.fc = nn.Linear(latent_dim, 64 * (output_dim // 8))
        self.convtrans1 = nn.ConvTranspose1d(64, 32, kernel_size=4, stride=2, padding=1)
        self.convtrans2 = nn.ConvTranspose1d(32, 16, kernel_size=4, stride=2, padding=1)
        self.convtrans3 = nn.ConvTranspose1d(16, 1, kernel_size=4, stride=2, padding=1)
        self.relu = nn.ReLU()
        self.output_activation = nn.Identity()

    def forward(self, z):
        z = self.fc(z)
        z = z.view(-1, 64, z.size(1) // 64)
        z = self.relu(self.convtrans1(z))
        z = self.relu(self.convtrans2(z))
        z = self.output_activation(self.convtrans3(z))
        return z

class LSEMVAE(nn.Module):
    def __init__(self, prior_dist, latent_dim, num_leads=12, input_dim_per_lead=5000):
        super(LSEMVAE, self).__init__()
        self.encoders = nn.ModuleList([ECGLeadEncoder(input_dim=input_dim_per_lead, latent_dim=latent_dim) for _ in range(num_leads)])
        self.shared_decoder = SharedDecoder(latent_dim=latent_dim, output_dim=input_dim_per_lead)
        self.pz = prior_dist
        self.poe = ProductOfExperts()
        self.latent_dim = latent_dim

    def sample_latent(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, inputs):
        qz_x_list, mus, logvars = [], [], []
        for encoder, input_signal in zip(self.encoders, inputs):
            mu, logvar = encoder(input_signal)
            qz_x_list.append((mu, logvar))
            mus.append(mu)
            logvars.append(logvar)

        prior_mu, prior_logvar = prior_expert(mus[0].shape, use_cuda=torch.cuda.is_available())
        prior_mu = prior_mu.to(mus[0].device)
        prior_logvar = prior_logvar.to(logvars[0].device)
        mus.append(prior_mu)
        logvars.append(prior_logvar)

        mus = torch.stack(mus)
        logvars = torch.stack(logvars)

        num_experts = mus.shape[0]
        num_groups = min(4, num_experts)
        group_size = num_experts // num_groups

        poe_mus, poe_logvars = [], []
        for i in range(num_groups):
            subset_mus = mus[i * group_size:(i + 1) * group_size]
            subset_logvars = logvars[i * group_size:(i + 1) * group_size]
            mu_poe, logvar_poe = self.poe(subset_mus, subset_logvars)
            poe_mus.append(mu_poe)
            poe_logvars.append(logvar_poe)

        poe_mus = torch.stack(poe_mus)
        poe_logvars = torch.stack(poe_logvars)
        mu_moe, logvar_moe = mixture_of_experts(poe_mus, poe_logvars)

        z_sample = self.sample_latent(mu_moe, logvar_moe)
        recon_leads = [self.shared_decoder(z_sample) for _ in range(len(self.encoders))]
        return qz_x_list, recon_leads, [z_sample]


In [ ]:
class ECGLeadClassifier(nn.Module):
    def __init__(self, pretrained_mopoe, num_classes=2, use_12_leads=False):
        super(ECGLeadClassifier, self).__init__()
        self.use_12_leads = use_12_leads
        self.lead_names = (
            ["LEAD_I", "LEAD_II", "LEAD_III", "LEAD_aVR", "LEAD_aVF", "LEAD_aVL", "LEAD_V1", "LEAD_V2", "LEAD_V3", "LEAD_V4", "LEAD_V5", "LEAD_V6"]
            if use_12_leads else
            ["LEAD_I", "LEAD_II", "LEAD_III", "LEAD_aVR", "LEAD_aVF", "LEAD_aVL"]
        )
        encoder_indices = range(12) if use_12_leads else range(6)
        self.lead_encoders = nn.ModuleList([pretrained_mopoe.encoders[i] for i in encoder_indices])
        self.feature_dim = pretrained_mopoe.latent_dim

        for encoder in self.lead_encoders:
            for param in encoder.parameters():
                param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(len(self.lead_names) * self.feature_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, lead_data):
        lead_features = []
        for lead_name, encoder in zip(self.lead_names, self.lead_encoders):
            mu, _ = encoder(lead_data[lead_name])
            lead_features.append(mu)
        combined_features = torch.cat(lead_features, dim=1)
        logits = self.classifier(combined_features)
        return logits

class BaseDataset(Dataset):
    def __init__(self, ecg_leads, labels):
        self.ecg_leads = ecg_leads
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        lead_data = {lead: self.ecg_leads[lead][idx].unsqueeze(0) for lead in self.ecg_leads}
        label = self.labels[idx]
        return lead_data, label

class NoisySubset(Dataset):
    def __init__(self, base_dataset, indices, severity=None, leads_to_noise=None, fs=500.0, f_bw=0.33, f_pl=50.0):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.severity = severity
        self.leads_to_noise = leads_to_noise or []
        self.fs = fs
        self.f_bw = f_bw
        self.f_pl = f_pl

    def __len__(self):
        return len(self.indices)

    def _inject(self, signal, snr_db, abw_amp, apl_amp):
        x = signal.squeeze(0).clone()
        n = x.numel()
        t = torch.arange(n, dtype=x.dtype, device=x.device) / self.fs
        phi_bw = torch.rand(1, device=x.device).item() * 2.0 * math.pi
        phi_pl = torch.rand(1, device=x.device).item() * 2.0 * math.pi
        bw = abw_amp * torch.sin(2.0 * math.pi * self.f_bw * t + phi_bw)
        pl = apl_amp * torch.sin(2.0 * math.pi * self.f_pl * t + phi_pl)
        x_art = x + bw + pl

        signal_power = torch.mean(x ** 2)
        noise_power = signal_power / (10.0 ** (snr_db / 10.0))
        eps = torch.randn_like(x)
        eps = eps - eps.mean()
        eps_std = eps.std(unbiased=False) + 1e-8
        eps = eps / eps_std
        eps = eps * torch.sqrt(noise_power + 1e-12)

        x_noisy = x_art + eps
        return x_noisy.unsqueeze(0)

    def __getitem__(self, i):
        idx = self.indices[i]
        lead_data, label = self.base_dataset[idx]

        if self.severity is not None:
            snr_db = self.severity["SNR"]
            abw_amp = self.severity["ABW"]
            apl_amp = self.severity["APL"]
            for lead in self.leads_to_noise:
                lead_data[lead] = self._inject(lead_data[lead], snr_db, abw_amp, apl_amp)

        return lead_data, label

def load_task_dataset(task_cfg):
    ecg_lead_tensors = {lead: torch.load(path) for lead, path in task_cfg["lead_file_paths"].items()}
    labels = torch.load(task_cfg["labels_file_path"])
    sample_count = len(next(iter(ecg_lead_tensors.values())))
    assert len(labels) == sample_count
    for tensor in ecg_lead_tensors.values():
        assert len(tensor) == sample_count
    return BaseDataset(ecg_lead_tensors, labels)

def load_pretrained_lsemvae():
    params = {"latent_dim": 256, "input_dim_per_lead": 5000, "num_leads": 12}
    prior_dist = prior_expert(params["latent_dim"])
    pretrained_mopoe = LSEMVAE(
        prior_dist=prior_dist,
        latent_dim=params["latent_dim"],
        num_leads=params["num_leads"],
        input_dim_per_lead=params["input_dim_per_lead"]
    )
    state_dict = torch.load(PRETRAINED_MODEL_PATH, map_location=device)
    new_state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
    pretrained_mopoe.load_state_dict(new_state_dict, strict=False)
    pretrained_mopoe.to(device)
    return pretrained_mopoe


In [ ]:
def train_classifier(model, train_loader, criterion, optimizer, epochs=50, use_12_leads=False):
    model.train()
    lead_names = (
        ["LEAD_I", "LEAD_II", "LEAD_III", "LEAD_aVR", "LEAD_aVF", "LEAD_aVL", "LEAD_V1", "LEAD_V2", "LEAD_V3", "LEAD_V4", "LEAD_V5", "LEAD_V6"]
        if use_12_leads else
        ["LEAD_I", "LEAD_II", "LEAD_III", "LEAD_aVR", "LEAD_aVF", "LEAD_aVL"]
    )

    for epoch in range(epochs):
        total_loss = 0.0
        for lead_data, labels in train_loader:
            labels = labels.to(device).long()
            lead_data = {lead: lead_data[lead].to(device) for lead in lead_names}
            optimizer.zero_grad()
            outputs = model(lead_data)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
        avg_loss = total_loss / len(train_loader.dataset)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

def evaluate_model(model, data_loader, use_12_leads=False):
    model.eval()
    all_labels = []
    all_probs = []
    all_preds = []

    lead_names = (
        ["LEAD_I", "LEAD_II", "LEAD_III", "LEAD_aVR", "LEAD_aVF", "LEAD_aVL", "LEAD_V1", "LEAD_V2", "LEAD_V3", "LEAD_V4", "LEAD_V5", "LEAD_V6"]
        if use_12_leads else
        ["LEAD_I", "LEAD_II", "LEAD_III", "LEAD_aVR", "LEAD_aVF", "LEAD_aVL"]
    )

    with torch.no_grad():
        for lead_data, labels in data_loader:
            labels = labels.to(device).long()
            lead_data = {lead: lead_data[lead].to(device) for lead in lead_names}
            logits = model(lead_data)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    auc_score = roc_auc_score(all_labels, all_probs)
    mcc = matthews_corrcoef(all_labels, all_preds)
    return accuracy, auc_score, mcc

def summarize_fold_metrics(metrics):
    arr = np.array(metrics)
    mean_vals = arr.mean(axis=0)
    std_vals = arr.std(axis=0)
    return {
        "Accuracy_mean": float(mean_vals[0]),
        "Accuracy_std": float(std_vals[0]),
        "AUROC_mean": float(mean_vals[1]),
        "AUROC_std": float(std_vals[1]),
        "MCC_mean": float(mean_vals[2]),
        "MCC_std": float(std_vals[2])
    }


In [ ]:
def run_noise_robustness_for_task(task_name, task_cfg, n_splits=5, epochs=50, batch_size=32, use_12_leads=False):
    base_dataset = load_task_dataset(task_cfg)
    labels = base_dataset.labels.numpy() if torch.is_tensor(base_dataset.labels) else np.array(base_dataset.labels)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    severity_fold_metrics = {name: [] for name in NOISE_LEVELS.keys()}

    for fold, (train_ids, test_ids) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        print(f"{task_name} Fold {fold}")
        train_subset = Subset(base_dataset, train_ids)
        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)

        pretrained_mopoe = load_pretrained_lsemvae()
        model = ECGLeadClassifier(pretrained_mopoe=pretrained_mopoe, num_classes=2, use_12_leads=use_12_leads).to(device)

        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
        criterion = nn.CrossEntropyLoss()

        train_classifier(model, train_loader, criterion, optimizer, epochs=epochs, use_12_leads=use_12_leads)

        for sev_name, sev_cfg in NOISE_LEVELS.items():
            noisy_test_subset = NoisySubset(
                base_dataset=base_dataset,
                indices=test_ids,
                severity=sev_cfg,
                leads_to_noise=LEADS_6,
                fs=FS,
                f_bw=F_BW,
                f_pl=F_PL
            )
            noisy_test_loader = DataLoader(noisy_test_subset, batch_size=batch_size, shuffle=False)
            acc, auc, mcc = evaluate_model(model, noisy_test_loader, use_12_leads=use_12_leads)
            severity_fold_metrics[sev_name].append((acc, auc, mcc))
            print(f"{task_name} Fold {fold} {sev_name}: Acc={acc:.4f}, AUROC={auc:.4f}, MCC={mcc:.4f}")

    summary = {sev: summarize_fold_metrics(vals) for sev, vals in severity_fold_metrics.items()}
    return summary


In [ ]:
all_results = {}
for task_name, task_cfg in TASK_CONFIGS.items():
    print(f"Running robustness for {task_name}")
    all_results[task_name] = run_noise_robustness_for_task(
        task_name=task_name,
        task_cfg=task_cfg,
        n_splits=5,
        epochs=50,
        batch_size=32,
        use_12_leads=False
    )

print(all_results)


In [ ]:
for task_name, task_result in all_results.items():
    print(f"\n{task_name}")
    for sev_name, metrics in task_result.items():
        print(
            sev_name,
            f"Acc {metrics['Accuracy_mean']:.4f}?{metrics['Accuracy_std']:.4f}",
            f"AUROC {metrics['AUROC_mean']:.4f}?{metrics['AUROC_std']:.4f}",
            f"MCC {metrics['MCC_mean']:.4f}?{metrics['MCC_std']:.4f}"
        )
